# 11 — End-to-End Pipeline

Full pipeline: data prep → tokenizer → model → training → generation → evaluation.
This notebook runs a small-scale version of the entire GPT pipeline.

## 1. Setup and Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
from src.config import GPTConfig, TrainConfig, SFTConfig
from src.data.preprocessor import DataPreprocessor
from src.tokenizer.bpe_tokenizer import BPETokenizer
from src.model.transformer import GPTModel
from src.training.pretrain import PretrainPipeline, set_seed
from src.training.finetune import SFTPipeline
from src.generation.generator import TextGenerator
from src.evaluation.metrics import Evaluator
from src.evaluation.error_analysis import ErrorAnalyzer

set_seed(42)
print('All imports OK')

## 2. Data Preprocessing

In [ ]:
dp = DataPreprocessor(normalize_arabic=True)

# Load and parse SFT data for a small demo corpus
sft_records = dp.parse_sft_data('../data/finetune/arabic_sft.jsonl')
stories = dp.parse_sft_data('../data/finetune/story_completion/arabic_stories.jsonl')
poetry = dp.parse_sft_data('../data/finetune/poetry/arabic_poetry.jsonl')

all_records = sft_records + stories + poetry
print(f'Total SFT records: {len(all_records)}')

# Build a small corpus from the SFT data for pretraining demo
corpus = '\n'.join(
    r['instruction'] + ' ' + r.get('input', '') + ' ' + r['output']
    for r in all_records
)
stats = dp.get_corpus_stats(corpus)
print(f"Corpus: {stats['char_count']:,} chars, {stats['byte_size']:,} bytes")

## 3. Tokenizer Training

In [ ]:
tok = BPETokenizer(vocab_size=500)
tok.train(corpus)
print(f'Vocabulary size: {len(tok.vocab)}')

# Quick encode/decode test
sample = 'مرحبا بالعالم'
ids = tok.encode(sample)
print(f'Encode/decode round-trip: {tok.decode(ids) == sample}')

## 4. Model Creation

In [ ]:
config = GPTConfig(
    vocab_size=len(tok.vocab),
    d_model=64,
    num_heads=4,
    num_layers=2,
    max_seq_len=128,
    dropout_rate=0.1,
)
config.validate()
model = GPTModel(config)
print(f'Parameters: {model.count_parameters():,}')

## 5. Pretraining (Small Scale)

In [ ]:
train_config = TrainConfig(
    learning_rate=3e-4,
    batch_size=4,
    max_steps=50,
    warmup_steps=5,
    log_interval=10,
    eval_interval=25,
    save_interval=100,  # won't trigger in 50 steps
    seed=42,
)
pipeline = PretrainPipeline(model, train_config, tok)
results = pipeline.train(corpus=corpus)
print(f'Train losses: {len(results["train_losses"])} entries')
print(f'Final train loss: {results["train_losses"][-1]:.4f}')

## 6. Text Generation (Pre-trained)

In [ ]:
model.eval()
gen = TextGenerator(model, tok)

prompts = ['في يوم من', 'كان هناك', 'The model']
for p in prompts:
    text, _ = gen.generate(p, max_new_tokens=20, temperature=0.8)
    print(f'Prompt: {p}')
    print(f'Output: {text}\n')

## 7. Supervised Fine-Tuning (Small Scale)

In [ ]:
sft_config = SFTConfig(
    learning_rate=1e-5,
    batch_size=4,
    max_steps=30,
    log_interval=10,
    eval_interval=15,
    save_interval=100,
    seed=42,
)
sft_pipeline = SFTPipeline(model, sft_config, tok)
sft_results = sft_pipeline.train(all_records[:10])  # use subset
print(f'SFT train losses: {len(sft_results["train_losses"])} entries')

## 8. Generation After Fine-Tuning

In [ ]:
model.eval()
for p in prompts:
    text, _ = gen.generate(p, max_new_tokens=20, temperature=0.8)
    print(f'Prompt: {p}')
    print(f'Output: {text}\n')

## 9. Evaluation

In [ ]:
evaluator = Evaluator()

# Perplexity from last val loss
if results['val_losses']:
    ppl = evaluator.perplexity_from_loss(results['val_losses'][-1])
    print(f'Perplexity: {ppl:.2f}')

# Error analysis on generated samples
samples = [gen.generate(p, max_new_tokens=30)[0] for p in prompts]
analyzer = ErrorAnalyzer()
report = analyzer.analyze(samples)
print(f'Error analysis: {report}')

## Summary

This notebook demonstrated the complete pipeline:
1. Data preprocessing and SFT parsing
2. BPE tokenizer training
3. GPT model creation
4. Pretraining with next-token prediction
5. Text generation
6. Supervised fine-tuning
7. Post-SFT generation
8. Evaluation (perplexity + error analysis)